<a href="https://colab.research.google.com/github/umais-developer/DATA622/blob/main/Copy_of_DATA622_Spring2026_Lab8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATA622 Lab 8 - Financial Phrasebank (Option 2)

**Student name:** Umais Siddiqui

This notebook runs Option 2 (unfreezing BERT layers) with Optuna for hyperparameter tuning. It is set up to run on a T4 GPU in Google Colab.

## Overview

In Lab 8, we will be working with the Financial Phrasebank dataset that we explored in the vignettes, which you can find here [Pretrained Vignettes](https://colab.research.google.com/drive/1pvc99evah2QEWWtTO6tj6luiZR_sq4mr?usp=sharing). This dataset consists of a series of snippets of text, ranging from roughly 10 to 90 words, describing an event that took place in the financial markets. This dataset is difficult for standard sentiment analysis tools because of the nuances of the languaged used by financial analysts. The dataset was annotated by experts, and we are using a subset of the dataset where the expert annotations all agreed. There are three classes in the dataset, 'positive', 'negative', and 'neutral', exemplified by the following examples:

<table>
  <thead>
    <tr>
      <th style="text-align:left">Text</th>
      <th style="text-align:center">Sentiment</th>
      <th style="text-align:center">Value</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Finnish Bank of Åland reports operating profit<br>of EUR 2.2 mn in the first quarter of 2010,<br>down from EUR 6.3 mn in the corresponding<br>period in 2009.</td>
      <td style="text-align:center">Negative</td>
      <td style="text-align:center">0</td>
    </tr>
    <tr>
      <td>The Samsung Mobile Applications Store was<br>launched in January 2009 by Samsung Mobile<br>Innovator, a program which enables mobile<br>software developers to create applications for<br>use across Samsung mobile devices.</td>
      <td style="text-align:center">Neutral</td>
      <td style="text-align:center">1</td>
    </tr>
    <tr>
      <td>Savon koulutuskuntayhtymä, Finland based<br>company has awarded contract for specialist<br>agricultural or forestry machinery.</td>
      <td style="text-align:center">Positive</td>
      <td style="text-align:center">2</td>
    </tr>
  </tbody>
</table>

You can read more details about this dataset on hugging faces: [FinanceMTEB](https://huggingface.co/datasets/FinanceMTEB/financial_phrasebank), or in the following paper:

1. Malo, P., Sinha, A., Korhonen, P., Wallenius, J., & Takala, P. (2014). "Good debt or bad debt: Detecting semantic orientations in economic texts." *Journal of the Association for Information Science and Technology*, *65*, 782–796.

## Problem 1

This assignment consists of a single problem. You have three different options, each based on training
and tuning a class of models with 'optuna' to understand both how tuning can improve the results of training, but also to understand the how optimal values of hyperparameters vary with model complexity and training architecture. The three options range according to a trade-off between computational difficulty and model power. You may select between them based on your interests and available compute. The first two options are possible with T4 GPUs (with the first one potentially requiring an hour long computation), but the third one is much more challenging and may require 4-plus hours on a T4 and the savvy needed to make sure everything is correct before starting the run.

The code examples in this week's vignette can be modified directly to solve the computational part of this problem. Read the extra credit first to see if you want to do it because you will want to design your simulations with it in mind.

I recommend you take the same approach that I did in the vignette with respect to training, testing, and validation, which is to split the test 50/50 set into a validation set and testing set and use the entire training set for training. This gives you a roughly 1\% standard error on your test accuracy and let's you maximize the amount of training data.

### Option 1: Finding Diminishing Returns for LSTM

In this option, you will look at how the test accuracy and best hyperparameters vary based on network width. Perform a loop over network widths starting at 4 and increasing rapidly, e.g. (say 32, 4, 8, 32, 256, and 1028) and a single network depth (1, 2, or 3 LSTM layers). For each width, perform an 'optuna' study to find the best values of the following parameters: 'learning rate', 'weight decay', 'epochs', and 'dropout'. You may use early stopping with a 'patience' parameter in-lieu of 'epochs'. Report how the optimal hyperparameters vary depending on network width, and also discuss the model performance. Perform between 10 and 20 trials in each study. Describe the general pattern that you observe. What do the results tell you about the importance of the embeddings versus the LSTM architecture? Report what you observe not what you expected to see.

Tip: For the highest accuracy, use BERT embeddings. To make that run fast, "pre-load" those embeddings by transforming the data using them. This prevents fine-tuning the embedding but prevents a costly loading step. This is discussed in the vignette notebook.

### Option 2: Unfreezing BERT

In this option, you will look at how test accuracy and optimal hyperparameters vary based on the number of unfrozen BERT layers. Perform a loop over the number of unfrozen layers, (I recommend '[0,2,4,8,12]', but you can tune based on your ambition and compute resources), and for each selection of unfrozen layers, perform an 'optuna' study to find the best accuracy and training hyperparameters. For the 'optuna' study, vary the 'learning rate' and 'weight decay'. I recommend using early stopping with patience=2 and a maximum of 10 epochs for training. Report your results. At what point does diminishing returns kick in for adding layers? What patterns do you observe in terms of how the best hyperparameters vary with number of layers? Report what you observe not what you expected to see.



### Option 3: Fine-Tuning Qwen 2.5

In this option, you will be fine tuning a small (0.5B parameters) open weights model called Qwen 2.5. Here you have the option of unfreezing different elements of each transformer block of Qwen. [This article](https://reintech.io/blog/how-to-fine-tune-llms-with-lora-and-qlora-practical-guide) suggests three levels of fine tuning based on the targeted modules: a minimal level which includes just 'q_proj' and 'v_proj' (the question and value layers), an all-attention level which also includes 'k_proj', and 'o_proj', and an all-linear level which includes all of the above and 'gate_proj', 'up_proj', and 'down_proj'. If you are interested, you may also test the all-MLP level, which is just 'gate_proj', 'up_proj', and 'down_proj'. For each of the levels, perform an 'optuna' study that finds the best values of the 'lora_r', the 'lora_dropout', the 'lora_alpha', and the 'learning_rate'. Train for 3-5 epochs for each trial. Are there meaningful differences in accuracy between the best trials for each level of freezing? What patterns do you observe with the optimal hyperparameters? For each of the best models identified by 'optuna', calculate the number of hyperparameters being tuned to find the fune-tuning prescription that is most efficient in terms of parameters. Report what you observe not what you expected to see.


### Extra Credit: Are some sentences intrinsically difficult? (5 Points)

To solve this problem, you must calculate the indices of incorrectly classified training examples either for the best model of each 'optuna' study or for the trained model of each 'optuna' trial. Calculate the accuracy for each question in the combined test and validation set across all the models. Is there evidence that some texts are intrinsically more difficult to classify? If so, print the 5 most difficult examples and hypothesize on why they are hard to classify.



In [ ]:
!pip -q install optuna datasets transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 28.8 MB/s eta 0:00:00


In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import optuna
from transformers import AutoTokenizer, AutoModel

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Switch runtime to a T4 GPU in Colab: Runtime -> Change runtime type -> GPU")

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
ds = load_dataset("FinanceMTEB/financial_phrasebank", split="train")

# If the dataset ships a combined table, filter to the 50agree subset when available.
subset_col = None
for candidate in ["subset", "configuration", "config", "dataset"]:
    if candidate in ds.column_names:
        subset_col = candidate
        break

if subset_col is not None:
    ds = ds.filter(lambda row: row[subset_col] == "sentences_50agree")

text_col = "sentence" if "sentence" in ds.column_names else "text"
texts = list(ds[text_col])
labels = list(ds["label"])

train_texts, holdout_texts, train_labels, holdout_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=42,
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    holdout_texts,
    holdout_labels,
    test_size=0.5,
    stratify=holdout_labels,
    random_state=42,
)

print("Train size:", len(train_texts))
print("Val size:", len(val_texts))
print("Test size:", len(test_texts))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/465 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/104k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/80.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1264 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train size: 1011
Val size: 126
Test size: 127


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
MAX_LEN = 128
BATCH_SIZE = 16

class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


def tokenize_texts(texts):
    return tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=MAX_LEN,
    )

train_enc = tokenize_texts(train_texts)
val_enc = tokenize_texts(val_texts)
test_enc = tokenize_texts(test_texts)
val_test_texts = val_texts + test_texts
val_test_labels = np.array(val_labels + test_labels)
val_test_enc = tokenize_texts(val_test_texts)

train_ds = TextDataset(train_enc, train_labels)
val_ds = TextDataset(val_enc, val_labels)
test_ds = TextDataset(test_enc, test_labels)
val_test_ds = TextDataset(val_test_enc, val_test_labels.tolist())

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
val_test_loader = DataLoader(val_test_ds, batch_size=BATCH_SIZE, shuffle=False)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
NUM_CLASSES = 3

class BertForSentiment(nn.Module):
    def __init__(self, model_name="bert-base-uncased", dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, NUM_CLASSES)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output
        if pooled is None:
            pooled = outputs.last_hidden_state[:, 0]
        logits = self.classifier(self.dropout(pooled))
        return logits


def set_trainable_layers(model, num_unfrozen_layers):
    for param in model.bert.parameters():
        param.requires_grad = False

    if num_unfrozen_layers > 0:
        for layer in model.bert.encoder.layer[-num_unfrozen_layers:]:
            for param in layer.parameters():
                param.requires_grad = True
        for param in model.bert.pooler.parameters():
            param.requires_grad = True

    for param in model.classifier.parameters():
        param.requires_grad = True

In [ ]:
MAX_EPOCHS = 10
PATIENCE = 2
EXTRA_CREDIT = True

misclass_counts = np.zeros(len(val_test_labels), dtype=np.int64)
total_models_trained = 0


def accuracy_from_logits(logits, labels):
    preds = torch.argmax(logits, dim=1)
    correct = (preds == labels).sum().item()
    return correct / len(labels)


def evaluate(model, loader):
    model.eval()
    total_correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(logits, dim=1)
            total_correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_correct / total


def predict_labels(model, loader):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(logits, dim=1)
            all_preds.append(preds.cpu().numpy())
    return np.concatenate(all_preds)


def train_one_run(num_unfrozen_layers, lr, wd, return_model=False):
    set_seed(42)
    model = BertForSentiment()
    set_trainable_layers(model, num_unfrozen_layers)
    model.to(device)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=wd,
    )
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    best_state = None
    patience_left = PATIENCE

    for _ in range(MAX_EPOCHS):
        model.train()
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        val_acc = evaluate(model, val_loader)
        if val_acc > best_val_acc + 1e-4:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    val_test_preds = None
    if EXTRA_CREDIT:
        val_test_preds = predict_labels(model, val_test_loader)

    if return_model:
        return best_val_acc, val_test_preds, model
    return best_val_acc, val_test_preds

In [ ]:
def make_objective(num_unfrozen_layers):
    def objective(trial):
        global misclass_counts, total_models_trained
        lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
        wd = trial.suggest_float("wd", 1e-6, 1e-2, log=True)

        val_acc, val_test_preds = train_one_run(num_unfrozen_layers, lr, wd)

        if EXTRA_CREDIT and val_test_preds is not None:
            misclass_counts += (val_test_preds != val_test_labels).astype(np.int64)
            total_models_trained += 1

        return val_acc

    return objective

In [ ]:
UNFREEZE_OPTIONS = [0, 2, 4, 8, 12]
N_TRIALS = 15

results = []

for num_unfrozen in UNFREEZE_OPTIONS:
    print(f"\n=== Unfrozen layers: {num_unfrozen} ===")
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=2),
    )
    study.optimize(make_objective(num_unfrozen), n_trials=N_TRIALS, show_progress_bar=True)

    best_params = study.best_params
    best_val = study.best_value

    best_val_acc, _, best_model = train_one_run(
        num_unfrozen, best_params["lr"], best_params["wd"], return_model=True
    )
    test_acc = evaluate(best_model, test_loader)

    results.append(
        {
            "unfrozen_layers": num_unfrozen,
            "best_val_acc": best_val,
            "test_acc": test_acc,
            "lr": best_params["lr"],
            "wd": best_params["wd"],
        }
    )

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("unfrozen_layers").reset_index(drop=True)
results_df

results_df.to_csv("lab8_option2_results.csv", index=False)
print("Saved results to lab8_option2_results.csv")


=== Unfrozen layers: 0 ===


  0%|          | 0/15 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved results to lab8_option2_results.csv


## Analysis (Option 2)

- **Accuracy vs. unfrozen layers:** <FILL IN: describe how validation and test accuracy changed as you unfroze more layers.>
- **Diminishing returns:** <FILL IN: identify where additional unfrozen layers stopped improving accuracy.>
- **Hyperparameter patterns:** <FILL IN: describe how the best learning rate and weight decay shifted across settings.>
- **Embedding vs. architecture:** <FILL IN: based on your results, comment on the importance of frozen embeddings vs. additional trainable layers.>

Remember: report what you observed, not what you expected to see.

In [ ]:
if EXTRA_CREDIT:
    if total_models_trained == 0:
        print("No models counted for extra credit. Check that EXTRA_CREDIT=True and studies ran.")
    else:
        difficulty = misclass_counts / total_models_trained
        hardest_idx = np.argsort(difficulty)[-5:][::-1]

        label_names = {0: "negative", 1: "neutral", 2: "positive"}
        print("Hardest 5 examples (val+test):")
        for rank, idx in enumerate(hardest_idx, 1):
            true_label = label_names[int(val_test_labels[idx])]
            print(f"\n#{rank} (difficulty={difficulty[idx]:.2f})")
            print(f"True label: {true_label}")
            print(val_test_texts[idx])

Hardest 5 examples (val+test):

#1 (difficulty=0.95)
True label: positive
This location makes the mall a convenient place to shop for consumers from three large residential areas nearby : Jaroszowka , Wysockiego and Zgody .

#2 (difficulty=0.92)
True label: negative
Frost sold shares for $ 19 million at $ 6.06-7 .12 per share , compared with Friday 's high of $ 11.33 and low of $ 10.14 .

#3 (difficulty=0.89)
True label: positive
MANAVIGATOR-September 7 , 2010-Kemira unveils Indian JV with IVRCL Finnish chemicals group Kemira ( HEL : KRA1V ) on Tuesday announced it has inked a deal to form a joint venture in India with local construction firm IVRCL Infrastructure and Projects Ltd ( BOM :530773 ) .

#4 (difficulty=0.84)
True label: positive
The company also said that in Poland a profitability program has been launched at the Oborniki steel frame and sandwich panel plant .

#5 (difficulty=0.77)
True label: negative
Operating profit was EUR 1.6 mn in 2005 compared to EUR 5.9 mn in 2004 .


## Extra Credit: Intrinsically Difficult Sentences

For each of the 5 hardest examples printed above, add 2-3 sentences hypothesizing why they are hard to classify (e.g., mixed signals, subtle wording, long sentences, or domain-specific jargon).

**Example 1:** The sentence reads like neutral location/retail convenience info with no explicit financial upside or downside. Words like “convenient” can imply positive sentiment, but in a business news context it is closer to neutral. The model likely confuses generic consumer‑friendly language with positive financial impact.

**Example 2:** The sentence is mostly about share pricing, which can be descriptive rather than evaluative. It compares a sale price to a prior high/low, which implies negative performance, but the direction is indirect. This kind of numeric comparison without explicit sentiment cues is often ambiguous.

**Example 3:** It announces a joint venture, which is usually positive, but the sentence is dense with proper nouns and ticker symbols. The business event is positive but understated; there is no explicit “growth” or “benefit” language. The model may treat it as neutral corporate news.

**Example 4:** “Profitability program” suggests improvement, but it is still a plan/initiative rather than a result. The sentence lacks concrete outcomes (e.g., higher profits), so it can be read as neutral. The model may not map restructuring language to positive sentiment.

**Example 5:** This is a decline in operating profit, which is negative, but the phrasing is purely comparative. If the model focuses on the presence of “profit” rather than the change, it can miss the negative direction. It is also short, so there are fewer sentiment cues.

